In [2]:
import numpy as np

In [43]:
x = np.array([1,2,-1,0])
w1 = np.array([[1,0,-1,2],[0,1,2,-1],[1,-2,0,1]])
b1 = np.array([0,1,-1])
w2 = np.array([[1,-1,2],[0,1,-1]])
b2 = np.array([0,1])
y = np.array([5,-1])
lr = 0.1
iter = 4

def relu(x):
    return np.maximum(0,x)

def d_relu(x):
    if x > 0:
        return 1
    return 0
    
def update_params(w1,w2,b1,b2,x,y,lr):
    z, yhat,h = forward(w1,w2,x,b1,b2)
    grad_yhat = grad_inp(yhat,y,h)
    w2l = grad_outp(grad_yhat, h)
    gh = back_prop(grad_yhat, w2)
    gz = apply_relud(gh, z)
    grad_w1l = first_layer_grad(gz, x)
    w1new, w2new, b1new, b2new = update_weights_and_biases(w1,w2,x,b1,b2,lr,grad_w1l,grad_yhat)
    return(w1new, w2new, b1new, b2new)

def forward(w1, w2, x, b1, b2):
    w1x = np.dot(w1,x.T)
    # print(w1x)
    z = w1x + b1
    h = relu(z)
    # print(h)
    w2h = np.dot(w2,h)
    # print(w2h)
    yhat = w2h + b2
    # print(yhat)
    return z, yhat, h

def grad_inp(yhat, y, h):
    l = (yhat - y)**2
    grad_yhat = 2*(yhat - y)
    # print(grad_yhat)
    return grad_yhat
    
def grad_outp(grad_yhat, h):
    w2l = np.outer(grad_yhat, h)
    # print(w2l)
    return w2l

def back_prop(grad_yhat, w2):
    gh = np.dot(w2.T,grad_yhat)
    # print(gh)
    return gh

def apply_relud(gh,z):
    d_relu_z = np.array([d_relu(i) for i in z])
    # print(d_relu_z)
    gz = gh * d_relu_z.T
    # print(gz)
    return gz

def first_layer_grad(gz, x):
    grad_w1l = np.outer(gz, x)
    # print(grad_w1l)
    return grad_w1l

def update_weights_and_biases(w1,w2,x,b1,b2,lr,grad_w1l,grad_yhat):
    w2new = w2-(lr*gz)
    w1new = w1 - (lr*grad_w1l)
    b1new = b1 - (lr*gz)
    b2new = b2 - (lr*grad_yhat)
    print('W2_New', w2new)
    print('W1_New', w1new)
    print('b1_New', b1new)
    print('b2_New', b2new)
    return w1new, w2new, b1new, b2new

for i in range(iter):
    print(f'Iteration {i}:\n')
    w1, w2, b1, b2 = update_params(w1,w2,b1,b2,x,y,lr)
    print('\n')


Iteration 0:

W2_New [[ 1.8 -2.4  2. ]
 [ 0.8 -0.4 -1. ]]
W1_New [[ 1.8  1.6 -1.8  2. ]
 [-1.4 -1.8  3.4 -1. ]
 [ 1.  -2.   0.   1. ]]
b1_New [ 0.8 -0.4 -1. ]
b2_New [0.8 0.4]


Iteration 1:

W2_New [[ 2.6 -3.8  2. ]
 [ 1.6 -1.8 -1. ]]
W1_New [[-2.8096 -7.6192  2.8096  2.    ]
 [-1.4    -1.8     3.4    -1.    ]
 [ 1.     -2.      0.      1.    ]]
b1_New [ 1.6 -1.8 -1. ]
b2_New [-1.096 -1.096]


Iteration 2:

W2_New [[ 3.4 -5.2  2. ]
 [ 2.4 -3.2 -1. ]]
W1_New [[-2.8096 -7.6192  2.8096  2.    ]
 [-1.4    -1.8     3.4    -1.    ]
 [ 1.     -2.      0.      1.    ]]
b1_New [ 2.4 -3.2 -1. ]
b2_New [ 0.1232 -1.0768]


Iteration 3:

W2_New [[ 4.2 -6.6  2. ]
 [ 3.2 -4.6 -1. ]]
W1_New [[-2.8096 -7.6192  2.8096  2.    ]
 [-1.4    -1.8     3.4    -1.    ]
 [ 1.     -2.      0.      1.    ]]
b1_New [ 3.2 -4.6 -1. ]
b2_New [ 1.09856 -1.06144]




In [71]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import *
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

train_data = pd.read_csv('./Data/diabetes_train.csv')
test_data = pd.read_csv('./Data/diabetes_test.csv')

def train_simple_network_bce_logits(model, loss_func, training_loader, epochs=20, device="cpu"):
    #Yellow step is done here. We create the optimizer and move the model to the compute device
    #SGD is Stochastic Gradient Decent over the parameters $\Theta$
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

    #Place the model on the correct compute resource (CPU or GPU)
    model.to(device)
    #The next two for loops handle the Red steps, iterating through all the data (batches) multiple times (epochs)
    for epoch in tqdm(range(epochs), desc="Epoch"):

        model = model.train()#Put our model in training mode
        running_loss = 0.0

        for inputs, labels in tqdm(training_loader, desc="Batch", leave=False):
            #Move the batch of data to the device we are using. this is the last red step
            inputs = inputs.to(device)
            labels = labels.to(device)

            #First a yellow step, prepare the optimizer. Most PyTorch code will do this first to make sure everything is in a clean and ready state.

            #PyTorch stores gradients in a mutable data structure. So we need to set it to a clean state before we use it.
            #Otherwise, it will have old information from a previous iteration
            optimizer.zero_grad()

            #The next two lines of code perform the two blue steps
            y_hat = model(inputs) #this just computed $f_\theta(\boldsymbol{x_i})
            print(y_hat.shape, labels.shape)

            # Compute loss.
            loss = loss_func(y_hat.view(-1,1), labels.view(-1,1))

            #Now the remaining two yellow steps, compute the gradient and ".step()" the optimizer
            loss.backward()# $\nabla_\Theta$ just got computed by this one call

            #Now we just need to update all the parameters
            optimizer.step()# $\Theta_{k+1} = \Theta_k − \eta \cdot \nabla_\Theta \ell(\hat{y}, y)$

            #Now we are just grabbing some information we would like to have
            running_loss += loss.item()
        # if epoch==1:
        #   print(y_hat, nn.functional.softmax(y_hat, dim=1))
        #   break
            
model = nn.Sequential(
    nn.Linear(8,  50),
    nn.ReLU(),
    nn.Linear(50,  50),
    nn.ReLU(),
    nn.Linear(50, 1),
)
print(train_data.shape)
train_data["label"] = train_data["class"].apply(lambda x: 1 if x == "tested_positive" else 0)
train_data.drop(columns=['class'], inplace=True)
X_train = torch.tensor(train_data.iloc[:, :-1].values, dtype=torch.float32)
y_train = torch.tensor(train_data.loc[:, 'label'].values, dtype=torch.float32)
train_dataset = TensorDataset(X_train, y_train)

training_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
loss_func = nn.BCEWithLogitsLoss()
train_simple_network_bce_logits(model, loss_func, training_loader, epochs=50)

(758, 9)


Batch:   0%|                                             | 0/24 [00:00<?, ?it/s]
                                                                                

torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([22, 1]) torch.Size([22])



Batch:   0%|                                             | 0/24 [00:00<?, ?it/s]
                                                                                

torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([22, 1]) torch.Size([22])


...

torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([22, 1]) torch.Size([22])



Batch:   0%|                                             | 0/24 [00:00<?, ?it/s]

torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])



Epoch: 100%|████████████████████████████████████| 50/50 [00:01<00:00, 27.28it/s]

torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([32, 1]) torch.Size([32])
torch.Size([22, 1]) torch.Size([22])


In [72]:
test_data["label"] = test_data["class"].apply(lambda x: 1 if x == "tested_positive" else 0)
test_data.drop(columns=['class'], inplace=True)
X_test = torch.tensor(test_data.iloc[:, :-1].values, dtype=torch.float32)
y_test = torch.tensor(test_data.loc[:, 'label'].values, dtype=torch.float32)


In [73]:
import torch
from sklearn.metrics import classification_report
model.eval()

with torch.no_grad():
    outputs = model(X_test)           
    probs = torch.sigmoid(outputs)   
    preds = (probs >= 0.5).int()     

print(classification_report(y_test.int(), preds))

              precision    recall  f1-score   support

           0       1.00      0.71      0.83         7
           1       0.60      1.00      0.75         3

    accuracy                           0.80        10
   macro avg       0.80      0.86      0.79        10
weighted avg       0.88      0.80      0.81        10

